# N070 · 树的层序遍历与视图

**目标：** 用层边界区分深度信息和全局队列。

**先修：** N053, N069。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** BFS队列；层大小；锯齿遍历；侧视图；N叉树层序。

## 从问题到算法

层序遍历在处理当前层之前固定队列长度，恰好取出这一层节点；处理中新增的是下一层，不能混入当前层。锯齿遍历只改变每层展示方向，不改变图搜索语义；右视图是每层最右节点，不一定是不断沿right走得到的链。

## 最小实现

**本章接口：** `level_order(root)`、`zigzag_level_order(root)`、`right_side_view(root)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root

from collections import deque

def level_order(root):
    if not root: return []
    queue=deque([root]); result=[]
    while queue:
        level=[]
        for _ in range(len(queue)):
            node=queue.popleft(); level.append(node.val)
            if node.left: queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level)
    return result

def zigzag_level_order(root):
    return [row if depth%2==0 else row[::-1] for depth,row in enumerate(level_order(root))]

def right_side_view(root):
    return [row[-1] for row in level_order(root)]

## 正确性、前提与复杂度

开始处理第d层时队列恰好包含该层节点，按左到右次序排列；每个节点的孩子追加到尾部，构成第d+1层的同样不变量。

**代价：** O(n)时间；level_order队列O(w)、输出O(n)。本章right_side_view复用全部层，额外保存O(n)，可用单独BFS优化到O(w)工作空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([3,9,20,None,None,15,7])
show_table(['深度','层节点','右侧可见'],[(d,row,row[-1]) for d,row in enumerate(level_order(r))])

深度,层节点,右侧可见
0,[3],3
1,"[9, 20]",20
2,"[15, 7]",7


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
r=tree_from_level([3,9,20,None,None,15,7])
assert level_order(r)==[[3],[9,20],[15,7]]
assert right_side_view(r)==[3,20,7]
assert zigzag_level_order(r)==[[3],[20,9],[15,7]]
assert level_order(None)==right_side_view(None)==[]
r=tree_from_level([1,2,None,3]); assert right_side_view(r)==[1,2,3]
print('N070: 所有本章断言通过')

N070: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 不反转结果列表实现锯齿。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释BFS辅助空间取决于最大层宽。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 102. Binary Tree Level Order Traversal（canonical）
- 103. Binary Tree Zigzag Level Order Traversal（canonical）
- 199. Binary Tree Right Side View（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。